In [3]:
import argparse
import numpy as np
import torch
from tqdm import tqdm
from eval_jux import Solver as eval_jux_solver
from sklearn.neighbors import KernelDensity

from util import ECFL


# set the random seed manually for reproducibility
SEED = 1
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

In [4]:
parser = argparse.ArgumentParser()


parser.add_argument('--device', default='cpu', type=str,
                    help='cpu/cuda')
parser.add_argument('--batch_size', default=1, type=int,
                    help='batch size')
parser.add_argument('--max_iter', default=0, type=float,
                    help='maximum number of batch iterations')
parser.add_argument('--ckpt_dir', default='ckpts', type=str)
parser.add_argument('--scale', default=1.0, type=float)
parser.add_argument('--num_goal', default=3, type=int)
parser.add_argument('--n_z', default=1, type=int)

# PFSD Params
parser.add_argument('--loader_num_workers', default=0, type=int)
parser.add_argument('--dt', default=0.4, type=float)
parser.add_argument('--obs_len', default=8, type=int)
parser.add_argument('--pred_len', default=12, type=int)
parser.add_argument('--dataset_dir', default='datasets/pfsd', type=str, help='dataset directory')
parser.add_argument('--dataset_name', default='pfsd', type=str,
                    help='dataset name')
parser.add_argument('--heatmap_size', default=160, type=int)
parser.add_argument('--diffusion_micro_cfg_id', default='pfsd_micro_diffuse', type=str)
parser.add_argument('--n_w', default=20, type=int)

args = parser.parse_args(args=[])

In [5]:
def compute_ECFL(output_traj, binary_navmaps):
    '''
    :param output_traj: (# scenes, # samples, # frames, # coordinates)
    :param binary_navmaps: (# scenes, # height/y, # width/x)
        1 indicates navigable; 0 indicates non-navigable
    :return: total collision count (for numerical stability, calulate the mean later)
    '''

    ecfl = 0.0
    for i in range(output_traj.shape[0]):
        for k in range(output_traj.shape[1]):
            collided = False
            for t in range(output_traj.shape[2]):
                pos = output_traj[i, k, t]
                if pos[1] < 0 or pos[1] >= binary_navmaps[i].shape[1] or pos[0] < 0 or pos[0] >= binary_navmaps[i].shape[0]:
                    collided = True

                    break
                if binary_navmaps[i][pos[0], pos[1]] == 0:
                    collided = True

                    break

            if not collided:
                ecfl += 1

    return round(ecfl, 2)

def PFSD_ECFL(solver, outputs):
     ECFL_total = 0
     data_iterator = iter(solver.data_loader)
    #  outputs = outputs.to(torch.double)
     for i, batch in tqdm(enumerate(data_iterator)):
          (obs_traj, fut_traj, obs_traj_st, fut_vel_st, seq_start_end,
               map_info, inv_h_t,
               local_map, local_ic, local_homo) = batch
        #   back_wc = torch.matmul(
        #                       torch.cat([outputs[:,i], torch.ones((outputs[:,i].shape[0], outputs[:,i].shape[1], 1)).to(outputs.device)], dim=2),
        #                       torch.transpose(torch.linalg.inv(local_homo[0]).float().to(outputs.device), 1, 0))
          back_wc = torch.matmul(
                torch.cat([outputs[:,i], torch.ones((outputs[:,i].shape[0], outputs[:,i].shape[1], 1)).to(outputs.device)], dim=2).double(),
                torch.transpose(torch.linalg.inv(local_homo[0]).float().to(outputs.device), 1, 0).double()
            )
          back_wc /= back_wc[:, :, 2].unsqueeze(-1)
          local_traj = back_wc[:,:,:2].int().unsqueeze(0)
          ECFL_instance = compute_ECFL(local_traj,(1-local_map).astype(int))
          ECFL_total += ECFL_instance
        #   print(ECFL_instance)
     return ECFL_total/(outputs.shape[0]*outputs.shape[1])


In [6]:
def DE_by_timestep(outputs, gt_futures):
    distance_to_gt = ((outputs-gt_futures)**2).sum(-1).sqrt()
    sorted_ade_mode_idx = distance_to_gt.mean(-1).sort(0).indices
    sorted_fde_mode_idx = distance_to_gt[...,-1].sort(0).indices
    cum_ade = []
    cum_fde = []
    for i in range(distance_to_gt.shape[-1]):
        cum_ade.append(torch.gather(distance_to_gt[...,:i+1].mean(-1), dim=0, index=sorted_ade_mode_idx)[0].mean().item())
        cum_fde.append(torch.gather(distance_to_gt[...,i], dim=0, index=sorted_fde_mode_idx)[0].mean().item())
    
    return cum_ade, cum_fde

In [7]:
import math
def kde_nll(outputs, gt_futures):
    log_dens = 0.0
    for i in tqdm(range(len(gt_futures))):
        X = outputs[:,i,:,:].cpu().numpy()
        Y = gt_futures[i,:,:].cpu().numpy()
        for t in range(len(Y)):
            kde = KernelDensity(kernel='gaussian', bandwidth=1).fit(X[:,t,:])
            log_dens += kde.score_samples(Y[t].reshape(1, -1)).item()

    ll = log_dens/gt_futures.shape[0]/gt_futures.shape[1]
    return -1*ll


In [8]:
# eval script for dataloader
solver = eval_jux_solver(args)
pfsd_20_gt = torch.load('Model_output_for_paper/pfsd/pfsd_gt.pt')

(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


/tmp/ipykernel_3586324/2877657269.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pfsd_20_gt = torch.load('Model_output_for_paper/pfsd/pfsd_gt.pt')


In [9]:
pfsd_20_gt.shape

torch.Size([4032, 12, 2])

In [10]:
# Trajectron++ PFSD 20
solver = eval_jux_solver(args)
trajectron_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_traj_pp.pt')
trajectron_pfsd_20_cum_ade, trajectron_pfsd_20_cum_fde = DE_by_timestep(trajectron_pfsd_20,pfsd_20_gt)
trajectron_pfsd_20_kde_nll = kde_nll(trajectron_pfsd_20,pfsd_20_gt)
trajectron_pfsd_20_ECFL = PFSD_ECFL(solver, trajectron_pfsd_20)
print("=====")
print("Trajectron++ PFSD 20")
print("ADE: ", trajectron_pfsd_20_cum_ade[-1])
print("FDE: ", trajectron_pfsd_20_cum_fde[-1])
print("KDE NLL: ", trajectron_pfsd_20_kde_nll)
print("ECFL: ", trajectron_pfsd_20_ECFL)
print("=====")

/tmp/ipykernel_3586324/3438718802.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  trajectron_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_traj_pp.pt')


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:19<00:00, 209.10it/s]
4032it [00:48, 82.39it/s]

=====
Trajectron++ PFSD 20
ADE:  0.19608939201027342
FDE:  0.42217574152182713
KDE NLL:  2.2442327179130976
ECFL:  0.8500496031746032
=====


In [19]:
# Agentformer PFSD 20
solver = eval_jux_solver(args)
agentformer_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_aformer.pt')
agentformer_pfsd_20_cum_ade, agentformer_pfsd_20_cum_fde = DE_by_timestep(agentformer_pfsd_20,pfsd_20_gt)
agentformer_pfsd_20_kde_nll = kde_nll(agentformer_pfsd_20,pfsd_20_gt)
agentformer_pfsd_20_ECFL = PFSD_ECFL(solver, agentformer_pfsd_20)
print("=====")
print("Agentformer PFSD 20")
print("ADE: ", agentformer_pfsd_20_cum_ade[-1])
print("FDE: ", agentformer_pfsd_20_cum_fde[-1])
print("KDE NLL: ", agentformer_pfsd_20_kde_nll)
print("ECFL: ", agentformer_pfsd_20_ECFL)
print("=====")

(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:14<00:00, 273.20it/s]
4032it [00:51, 78.63it/s]

=====
Agentformer PFSD 20
ADE:  0.11219748109579086
FDE:  0.16541415452957153
KDE NLL:  1.9369599519537761
ECFL:  0.9376364087301587
=====


In [20]:
import pickle5 as pickle
file_path = 'Model_output_for_paper/pfsd/pfsd_preprocess_ynet.pkl'
# file_path = 'nuscenes/10/nuscenes_10_ynet.pkl'
with open(file_path, 'rb') as file:
    check = pickle.load(file)
check.keys()
y_net = check['sample'].cpu()
y_net_gt = check['gt'].cpu()
ynet_split = check['seq_start_end']

In [ ]:
# torch.save(y_net, 'Model_output_for_paper/pfsd/pfsd_20_ynet.pt')

In [26]:
# Ynet PFSD 20
solver = eval_jux_solver(args)
ynet_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_ynet.pt')
# ynet_pfsd_20 = y_net
ynet_pfsd_20_cum_ade, ynet_pfsd_20_cum_fde = DE_by_timestep(ynet_pfsd_20,pfsd_20_gt)
ynet_pfsd_20_kde_nll = kde_nll(ynet_pfsd_20,pfsd_20_gt)
ynet_pfsd_20_ECFL = PFSD_ECFL(solver, ynet_pfsd_20)
print("=====")
print("Ynet PFSD 20")
print("ADE: ", ynet_pfsd_20_cum_ade[-1])
print("FDE: ", ynet_pfsd_20_cum_fde[-1])
print("KDE NLL: ", ynet_pfsd_20_kde_nll)
print("ECFL: ", ynet_pfsd_20_ECFL)
print("=====")


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:14<00:00, 273.33it/s]
4032it [00:50, 79.23it/s]

=====
Ynet PFSD 20
ADE:  0.07467690110206604
FDE:  0.12458166480064392
KDE NLL:  1.9877477986395267
ECFL:  0.9415674603174603
=====


In [22]:
# Muse PFSD 20
solver = eval_jux_solver(args)
muse_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_muse.pt')
muse_pfsd_20_cum_ade, muse_pfsd_20_cum_fde = DE_by_timestep(muse_pfsd_20,pfsd_20_gt)
muse_pfsd_20_kde_nll = kde_nll(muse_pfsd_20,pfsd_20_gt)
muse_pfsd_20_ECFL = PFSD_ECFL(solver, muse_pfsd_20)
print("=====")
print("Muse PFSD 20")
print("ADE: ", muse_pfsd_20_cum_ade[-1])
print("FDE: ", muse_pfsd_20_cum_fde[-1])
print("KDE NLL: ", muse_pfsd_20_kde_nll)
print("ECFL: ", muse_pfsd_20_ECFL)
print("=====")


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:15<00:00, 265.53it/s]
4032it [00:52, 77.06it/s]

=====
Muse PFSD 20
ADE:  0.05358647555112839
FDE:  0.08861290663480759
KDE NLL:  1.950121331126397
ECFL:  0.9708829365079366
=====


In [23]:
# Diffuse PFSD 20
solver = eval_jux_solver(args)
diffuse_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_diffuse.pt')
diffuse_pfsd_20_cum_ade, diffuse_pfsd_20_cum_fde = DE_by_timestep(diffuse_pfsd_20,pfsd_20_gt)
diffuse_pfsd_20_kde_nll = kde_nll(diffuse_pfsd_20,pfsd_20_gt)
diffuse_pfsd_20_ECFL = PFSD_ECFL(solver, diffuse_pfsd_20)
print("=====")
print("Diffuse PFSD 20")
print("ADE: ", diffuse_pfsd_20_cum_ade[-1])
print("FDE: ", diffuse_pfsd_20_cum_fde[-1])
print("KDE NLL: ", diffuse_pfsd_20_kde_nll)
print("ECFL: ", diffuse_pfsd_20_ECFL)
print("=====")


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:14<00:00, 269.45it/s]
4032it [00:51, 77.90it/s]

=====
Diffuse PFSD 20
ADE:  0.05960460379719734
FDE:  0.0900062620639801
KDE NLL:  1.9853733349728486
ECFL:  0.9853298611111111
=====


In [24]:
# Guided PFSD 20
solver = eval_jux_solver(args)
guided_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_guided.pt')
guided_pfsd_20_cum_ade, guided_pfsd_20_cum_fde = DE_by_timestep(guided_pfsd_20,pfsd_20_gt)
guided_pfsd_20_kde_nll = kde_nll(guided_pfsd_20,pfsd_20_gt)
guided_pfsd_20_ECFL = PFSD_ECFL(solver, guided_pfsd_20)
print("=====")
print("Guided PFSD 20")
print("ADE: ", guided_pfsd_20_cum_ade[-1])
print("FDE: ", guided_pfsd_20_cum_fde[-1])
print("KDE NLL: ", guided_pfsd_20_kde_nll)
print("ECFL: ", guided_pfsd_20_ECFL)
print("=====")


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:14<00:00, 272.85it/s]
4032it [00:51, 77.78it/s]

=====
Guided PFSD 20
ADE:  0.0598900206387043
FDE:  0.09009759873151779
KDE NLL:  1.9865557961199336
ECFL:  0.9962301587301587
=====


In [ ]:
# import pickle5 as pickle
# with open('Model_output_for_paper/pfsd/pfsd_20_MID.pkl', 'rb') as file:
#     check = pickle.load(file)

# MID_pfsd_20 = []
# for key in check.keys():
#     MID_pfsd_20.append(torch.from_numpy(check[key]['pred']))
# MID_pfsd_20 = torch.cat(MID_pfsd_20, dim=1)
# torch.save(MID_pfsd_20, 'Model_output_for_paper/pfsd/pfsd_20_MID.pt')

In [11]:
# MID PFSD 20
solver = eval_jux_solver(args)
MID_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_MID.pt')
MID_pfsd_20_cum_ade, MID_pfsd_20_cum_fde = DE_by_timestep(MID_pfsd_20,pfsd_20_gt)
MID_pfsd_20_kde_nll = kde_nll(MID_pfsd_20,pfsd_20_gt)
MID_pfsd_20_ECFL = PFSD_ECFL(solver, MID_pfsd_20)
print("=====")
print("MID PFSD 20")
print("ADE: ", MID_pfsd_20_cum_ade[-1])
print("FDE: ", MID_pfsd_20_cum_fde[-1])
print("KDE NLL: ", MID_pfsd_20_kde_nll)
print("ECFL: ", MID_pfsd_20_ECFL)
print("=====")


/tmp/ipykernel_3586324/476867807.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  MID_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_MID.pt')


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:19<00:00, 210.44it/s]
4032it [00:51, 78.03it/s]

=====
MID PFSD 20
ADE:  0.0925624743103981
FDE:  0.16301821172237396
KDE NLL:  2.00206146074541
ECFL:  0.8872395833333333
=====


In [ ]:
# import pickle5 as pickle
# with open('Model_output_for_paper/pfsd/pfsd_20_MID_map.pkl', 'rb') as file:
#     check = pickle.load(file)

# MID_map_pfsd_20 = []
# for key in check.keys():
#     MID_map_pfsd_20.append(torch.from_numpy(check[key]['pred']))
# MID_map_pfsd_20 = torch.cat(MID_map_pfsd_20, dim=1)
# torch.save(MID_map_pfsd_20, 'Model_output_for_paper/pfsd/pfsd_20_MID_map.pt')

In [11]:
# MID_map PFSD 20
solver = eval_jux_solver(args)
MID_map_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_MID_map.pt')
MID_map_pfsd_20_cum_ade, MID_map_pfsd_20_cum_fde = DE_by_timestep(MID_map_pfsd_20,pfsd_20_gt)
MID_map_pfsd_20_kde_nll = kde_nll(MID_map_pfsd_20,pfsd_20_gt)
MID_map_pfsd_20_ECFL = PFSD_ECFL(solver, MID_map_pfsd_20)
print("=====")
print("MID_map PFSD 20")
print("ADE: ", MID_map_pfsd_20_cum_ade[-1])
print("FDE: ", MID_map_pfsd_20_cum_fde[-1])
print("KDE NLL: ", MID_map_pfsd_20_kde_nll)
print("ECFL: ", MID_map_pfsd_20_ECFL)
print("=====")


(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:15<00:00, 256.27it/s]
4032it [00:56, 70.82it/s]

=====
MID_map PFSD 20
ADE:  0.10218475013971329
FDE:  0.1908353716135025
KDE NLL:  2.0036843924727057
ECFL:  0.9040550595238095
=====


In [11]:
# LED PFSD 20
solver = eval_jux_solver(args)
LED_pfsd_20 = torch.load('Model_output_for_paper/pfsd/pfsd_20_LED.pt').transpose(0,1)
LED_pfsd_20_cum_ade, LED_pfsd_20_cum_fde = DE_by_timestep(LED_pfsd_20,pfsd_20_gt)
LED_pfsd_20_kde_nll = kde_nll(LED_pfsd_20,pfsd_20_gt)
LED_pfsd_20_ECFL = PFSD_ECFL(solver, LED_pfsd_20)
print("=====")
print("LED PFSD 20")
print("ADE: ", LED_pfsd_20_cum_ade[-1])
print("FDE: ", LED_pfsd_20_cum_fde[-1])
print("KDE NLL: ", LED_pfsd_20_kde_nll)
print("ECFL: ", LED_pfsd_20_ECFL)
print("=====")

(4012, 4032)
[ models/temporal ] Channel dimensions: [(2, 16), (16, 64), (64, 128)]
[(2, 16), (16, 64), (64, 128)]


100%|██████████| 4032/4032 [00:17<00:00, 231.75it/s]
4032it [01:30, 44.40it/s]

=====
LED PFSD 20
ADE:  0.21877171099185944
FDE:  0.3366169035434723
KDE NLL:  2.3351707207795376
ECFL:  0.780592757936508
=====


In [9]:
LED_pfsd_20.shape

torch.Size([4032, 20, 12, 2])